# Sparse-cache continuation diagnostic, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Three frozen `plan4_dev` families, one per difficulty, including variants (9 examples):
`plan4_dev-fam-006` (L1), `plan4_dev-fam-001` (L2), `plan4_dev-fam-000` (L3).

A writes calculations once. B's instructions stay frozen. Compare:

- `kv`: full completed cache.
- `kv_identity`: that cache passed through the shared-position gather while keeping every index. Tensors should match; answers should match `kv`.
- `kv_result`: RESULT-line KV only (the 0/90 failure).
- `kv_result_sink`: RESULT-line KV plus the first four original cache positions. Count the extras.
- `text_result`: those RESULT lines as text.

Save raw outputs, retained positions, generate() token ids, pos_cursor, and mask length. Do not reset retained key positions.

Do **not** load `plan4_fresh`. Do not resume `results_stage_plan4_span`. No budgets 32/64 or OBF.

Span reference: full KV 82/90, result text 79/90, result KV 0/90, probe text 23/90, probe KV 21/90.

Download `/kaggle/working/results_stage_plan4_kvdiag`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_kvdiag.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_kvdiag.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_kvdiag.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_kvdiag.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_dev", sum(1 for _ in (REPO / "data" / "plan4_dev.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_kvdiag")
print("Three families. Identity gather, result KV, result KV plus first four positions, result text.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_kvdiag", "--split", "plan4_dev"])
print("plan4 kvdiag exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_kvdiag"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_kvdiag" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    print(path.read_text(encoding="utf-8")[:16000])